# 03 Gold - Build Skill Marts

This notebook mirrors the local dbt mart layer.

Local equivalents:

- `dbt/models/marts/monthly_skill_counts.sql`
- `dbt/models/marts/mart_dashboard_skill_trends.sql`
- `dbt/models/marts/mart_skill_geography.sql`

Fabric role:

- aggregate Silver skills by month
- normalize mentions as share of ads
- create dashboard/report-ready Gold Delta tables

In [ ]:
from pyspark.sql import functions as F

BRONZE_TABLE = "historical_job_ads_bronze"
SILVER_TABLE = "historical_regex_skills_silver"

MONTHLY_COUNTS_TABLE = "monthly_skill_counts_gold"
DASHBOARD_TRENDS_TABLE = "mart_dashboard_skill_trends_gold"
GEOGRAPHY_TABLE = "mart_skill_geography_gold"

ads = spark.table(BRONZE_TABLE)
skills = spark.table(SILVER_TABLE)

print(f"Loaded {BRONZE_TABLE} and {SILVER_TABLE}")

In [ ]:
# Gold mart 1: raw monthly skill mentions.
monthly_skill_counts = (
    skills
    .groupBy("publication_month", "skill")
    .agg(F.countDistinct("id").alias("mentions"))
)

monthly_skill_counts.write.mode("overwrite").format("delta").saveAsTable(MONTHLY_COUNTS_TABLE)

print(f"Wrote Gold table: {MONTHLY_COUNTS_TABLE}")

In [ ]:
# Gold mart 2: normalized dashboard trend table.
monthly_ads = (
    ads
    .groupBy("publication_month")
    .agg(F.countDistinct("id").alias("ads"))
)

dashboard_trends = (
    monthly_skill_counts
    .join(monthly_ads, on="publication_month", how="inner")
    .withColumn("share_of_ads", F.col("mentions") / F.col("ads"))
)

dashboard_trends.write.mode("overwrite").format("delta").saveAsTable(DASHBOARD_TRENDS_TABLE)

print(f"Wrote Gold table: {DASHBOARD_TRENDS_TABLE}")

In [ ]:
# Gold mart 3: skill demand by municipality. The Sweden bounding box mirrors
# the local dbt mart that removes invalid coordinates outside Sweden.
valid_geo_ads = ads.where(
    (F.col("longitude").between(10, 25))
    & (F.col("latitude").between(55, 70))
    & F.col("municipality").isNotNull()
)

geo_ads = (
    valid_geo_ads
    .groupBy("publication_month", "municipality", "municipality_code", "region", "region_code", "longitude", "latitude")
    .agg(F.countDistinct("id").alias("ads"))
)

geo_mentions = (
    skills
    .join(
        valid_geo_ads.select("id", "publication_month", "municipality", "municipality_code", "region", "region_code", "longitude", "latitude"),
        on=["id", "publication_month"],
        how="inner",
    )
    .groupBy("publication_month", "skill", "municipality", "municipality_code", "region", "region_code", "longitude", "latitude")
    .agg(F.countDistinct("id").alias("mentions"))
)

skill_geography = (
    geo_mentions
    .join(
        geo_ads,
        on=["publication_month", "municipality", "municipality_code", "region", "region_code", "longitude", "latitude"],
        how="inner",
    )
    .withColumn("share_of_ads", F.col("mentions") / F.col("ads"))
)

skill_geography.write.mode("overwrite").format("delta").saveAsTable(GEOGRAPHY_TABLE)

print(f"Wrote Gold table: {GEOGRAPHY_TABLE}")

In [ ]:
display(
    spark.table(DASHBOARD_TRENDS_TABLE)
    .orderBy(F.desc("publication_month"), F.desc("mentions"))
    .limit(50)
)